# Commodity prediction · evidence review

Presentation of saved aggregate results only. No model loading, fitting, raw-data access, or research-notebook execution. Run this notebook in the **review worktree** after the publication overlay is applied. Do not compare scores from different evaluation periods. Save and reopen to confirm inline outputs persist.

In [1]:
from pathlib import Path
import json
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
ROOT=Path.cwd()
if ROOT.name=='notebooks': ROOT=ROOT.parent
index=json.loads((ROOT/'reports/manual_research/index.json').read_text())
studies=index['studies']
pio.renderers.default='plotly_mimetype'
rows=[dict(notebook=s['notebook'],title=s['title'],origins=s['evaluation_origins'],**r) for s in studies if s['completed'] for r in s['comparisons']]
df=pd.DataFrame(rows)
display(pd.DataFrame([{k:s[k] for k in ('notebook','title','status','evaluation_origins')} for s in studies]))
def show(fig,title,x,y):
    fig.update_layout(title=title,xaxis_title=x,yaxis_title=y,height=560,margin=dict(l=80,r=30,t=90,b=130))
    fig.show()


,notebook,title,status,evaluation_origins
0,3,Runtime and feature readiness,NOTEBOOK_AND_FEATURE_AUDIT_READY,0
1,4,Normalization first-period screen,NOTEBOOK_AND_FIRST_FOLD_REVIEW_READY,180
2,5,Normalization temporal validation,NOTEBOOK_AND_VALIDATION_REVIEW_READY,535
3,6,Session candidate laboratory,NOTEBOOK_AND_SESSION_FEATURES_READY,0
4,7,Session feature ablations,NOTEBOOK_AND_SESSION_ABLATION_READY,180
5,8,Cross-asset peer features,NOTEBOOK_AND_CLOSE_NETWORK_READY,180
6,9,Network removal and rank laboratory,NOTEBOOK_AND_DIAGNOSIS_READY,180
7,10,Released historical-rank states,NOTEBOOK_AND_RANK_STATE_READY,180
8,11,Instrument context,NOTEBOOK_AND_TARGET_CONTEXT_READY,180
9,12,Delayed response encoding,NOTEBOOK_AND_RESPONSE_READY,180


## Historical reference across periods
The same fixed comparison, on three different chronological intervals. Its variability motivates replication.

In [2]:
f=go.Figure(go.Bar(x=['1169–1348','1349–1528','1529–1703'],y=[.40338108742296147,.16704165319061334,.39061886486364056]))
show(f,'1 · Historical reference by period','Development origins','Daily-correlation mean / population standard deviation')

## Matched improvements
Compare deltas within the evaluation used by each study. This chart does not pool incompatible predictions.

In [3]:
f=go.Figure()
if not df.empty:
    for n,g in df.groupby('notebook',sort=True):
        delta=g.get('delta_vs_current_market',g.get('matched_delta',pd.Series(index=g.index,dtype=float)))
        if 'matched_delta' in g: delta=delta.fillna(g['matched_delta'])
        f.add_bar(x=[f'{n:02d}: '+v for v in g['variant']],y=delta,name=f'Notebook {n:02d}')
f.add_hline(y=0)
show(f,'2 · Matched development changes — different periods remain separate','Experiment / panel','Within-study metric difference')

## Candidate replication
This section uses report 16 only. An absent report produces an explicit gap rather than a fabricated result.

In [4]:
r16=next(s for s in studies if s['notebook']==16)
f=go.Figure()
if r16['completed']:
    f.add_bar(x=[r['variant'] for r in r16['comparisons']],y=[r['official_metric'] for r in r16['comparisons']])
    f.add_hline(y=.3097087232124053,line_dash='dash')
else:
    f.add_annotation(text='Completed replication report not available in this release',showarrow=False)
show(f,'3 · Prior-dynamics replication, 535 origins only','Declared representation','Pooled development metric')

## Execution accounting
Fit counts describe the reported experiment cost; more fits are not evidence of better science.

In [5]:
r=[s for s in studies if s['completed'] and 'new_training_fits' in s]
f=go.Figure(go.Bar(x=[str(s['notebook']) for s in r],y=[s['new_training_fits'] for s in r]))
show(f,'4 · Reported new fits by milestone','Notebook','Reported new fits')

## Bounded analytical time
These are saved report timings, not an estimate of all cloud charges or all work ever performed.

In [6]:
r=[(s,next((s[k] for k in ('cumulative_supervised_seconds','supervised_seconds','elapsed_seconds') if k in s),None)) for s in studies if s['completed']]
r=[(s,t) for s,t in r if t is not None]
f=go.Figure(go.Bar(x=[str(s['notebook']) for s,t in r],y=[t for s,t in r]))
show(f,'5 · Recorded analytical/supervised time','Notebook','Seconds (report timing scope)')

## Completion and evidence gaps
Only available reports supply completion status. An unexecuted template is not an executed result.

In [7]:
counts=pd.Series([s['status'] for s in studies]).value_counts()
f=go.Figure(go.Bar(x=list(counts.index),y=counts.tolist()))
show(f,'6 · Evidence status across the manual research sequence','Recorded status','Number of milestones')
print('RESULT: PORTFOLIO_OVERVIEW_READY')
print('Save this notebook, reopen without running, and verify all six figures are visible.')

RESULT: PORTFOLIO_OVERVIEW_READY
Save this notebook, reopen without running, and verify all six figures are visible.
